# 02 · Baselines: TF-IDF + logistic regression, and a fastText-style bag of embeddings
Owner: **Member 1** · CPU is enough

Both baselines ignore word order (apart from short n-grams), so they answer: *how far do we get
without modelling the sequence?*

| Baseline | Order | Why include it |
|---|---|---|
| **TF-IDF (word + char n-grams) + logistic regression** | only within 1–2-word / 2–5-char windows | The classic, very strong text-classification baseline; character n-grams handle Swahili morphology. |
| **fastText-style bag of embeddings** (Joulin et al., 2017) | none (averaged word + hashed bigram vectors) | A *neural* model without sequence modelling: separates "being neural" from "modelling order". |

Selection uses **validation log loss** (official metric). The test split is used once at the end.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv` and `SampleSubmission.csv` (from Zindi) in a
Google Drive folder called `swahili_news`, and set `REPO_URL` below.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-news-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-news-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_news")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    for name in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        shutil.copy(DRIVE_DATA / name, REPO_DIR / "data" / name)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")             # e.g. downloaded word vectors persist
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")   # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from src import config as C
from src.data import get_splits, split_frames, encode_labels, load_unlabelled
from src.datasets import TokenDataset, pad_collate
from src.evaluate import compute_metrics, log_experiment, save_final_results, make_submission, show_log
from src.models import FastTextBag
from src.pipeline import run_experiments, finalize
from src.text import tokenize, head_tail, Vocab, encode_corpus, SW_STOPWORDS
from src.utils import set_seed, Timer

MEMBER = "Member 1"
set_seed(C.SEED)
df = get_splits()
tr, va, te = split_frames(df)
ytr, yva, yte = (encode_labels(d.label) for d in (tr, va, te))
zindi = load_unlabelled()
K = len(C.CLASSES)

## Part A · TF-IDF + logistic regression
Every variant is a function of the article text, so the same code builds each experiment.

In [ ]:
def tfidf_features(cfg, fit_texts, *other):
    prep = lambda t: " ".join(head_tail(tokenize(t), cfg.get("max_words", 10**6), cfg.get("head_frac", 1.0)))
    if cfg.get("tail_only"):
        prep = lambda t: " ".join(tokenize(t)[-cfg["max_words"]:])
    stop = SW_STOPWORDS if cfg.get("stopwords") else None
    vecs = [TfidfVectorizer(preprocessor=prep, tokenizer=str.split, token_pattern=None, lowercase=False,
                            ngram_range=(1, cfg.get("word_ngrams", 2)), min_df=2, max_features=200_000,
                            sublinear_tf=True, stop_words=list(stop) if stop else None)]
    if cfg.get("char"):
        vecs.append(TfidfVectorizer(preprocessor=prep, analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                                    max_features=300_000, sublinear_tf=True))
    Xs = [hstack([v.fit_transform(fit_texts) for v in vecs]).tocsr()]
    for texts in other:
        Xs.append(hstack([v.transform(texts) for v in vecs]).tocsr())
    return Xs


def run_lr(exp):
    cfg = exp["cfg"]
    Xtr, Xva = tfidf_features(cfg, tr.content, va.content)
    best = None
    for Cval in ([4.0] if C.QUICK else [1.0, 4.0, 16.0]):        # C tuned on validation log loss
        clf = LogisticRegression(C=Cval, max_iter=3000, class_weight="balanced" if cfg.get("balanced") else None)
        clf.fit(Xtr, ytr)
        m = compute_metrics(yva, clf.predict_proba(Xva), K)
        if best is None or m["log_loss"] < best[1]["log_loss"]:
            best = (Cval, m, clf)
    return best


LR_EXPS = [
    dict(id="BL-01", description="TF-IDF word 1-2 grams + LR", cfg={},
         hypothesis="Topic words alone separate most categories.", motivated_by="Starting point"),
    dict(id="BL-02", description="+ character 2-5 grams", cfg={"char": True},
         hypothesis="Char n-grams share evidence across inflected forms (mchezo, wachezaji, kucheza).",
         motivated_by="EDA 5: 53% of word types are hapax; many forms of one stem"),
    dict(id="BL-03", description="+ char n-grams + balanced class weights", cfg={"char": True, "balanced": True},
         hypothesis="Re-weighting improves recall on kimataifa/burudani (54 and 17 articles).",
         motivated_by="EDA 2: 118:1 imbalance; BL-02 per-class recall on minority classes"),
    dict(id="BL-04", description="word+char, FIRST 128 words only", cfg={"char": True, "max_words": 128},
         hypothesis="News puts the key facts first (inverted pyramid), so the head carries most signal.",
         motivated_by="EDA 8: keyword position; informs truncation choice for sequence models"),
    dict(id="BL-05", description="word+char, LAST 128 words only", cfg={"char": True, "max_words": 128, "tail_only": True},
         hypothesis="The end of an article is less informative than the head.",
         motivated_by="BL-04: compare head vs tail at the same length"),
    dict(id="BL-06", description="word+char, Swahili stop-words removed", cfg={"char": True, "stopwords": True},
         hypothesis="Function words are noise for topic classification.",
         motivated_by="Common preprocessing choice; checking whether it actually helps here"),
]
results_lr = {}
for e in LR_EXPS:
    with Timer() as t:
        Cval, m, clf = run_lr(e)
    results_lr[e["id"]] = (e, Cval, m)
    log_experiment(e["id"], MEMBER, "TF-IDF LR", e["description"], e["hypothesis"], {**e["cfg"], "C": Cval},
                   m, e["motivated_by"], t.seconds)
    print(f"{e['id']}: val log loss {m['log_loss']:.4f} | acc {m['accuracy']:.3f} | macro-F1 {m['macro_f1']:.3f} | C={Cval}")

## Part B · fastText-style bag of embeddings (neural, order-free)

In [ ]:
vocab = Vocab(tr.content.map(tokenize))
seq = {name: encode_corpus(d.content, vocab, max_len=10**6) for name, d in
       {"tr": tr, "va": va, "te": te, "z": zindi}.items()}
print("vocabulary size:", len(vocab))


def make_datasets(p):
    return TokenDataset(seq["tr"], ytr), TokenDataset(seq["va"], yva)


def build_model(p):
    return FastTextBag(len(vocab), K, dim=p.get("dim", 100), bigrams=p.get("bigrams", True))


FT_EXPS = [
    dict(id="FT-01", description="Averaged word embeddings (unigrams)", params={"bigrams": False},
         hypothesis="A learned dense bag of words matches TF-IDF.", motivated_by="Neural order-free reference"),
    dict(id="FT-02", description="+ hashed bigram embeddings", params={"bigrams": True},
         hypothesis="Bigrams add a little local order ('waziri wa', 'ligi kuu').", motivated_by="FT-01"),
]
best_ft, ft_summary = run_experiments(
    FT_EXPS, build_model, make_datasets, member=MEMBER, model_name="fastText bag", n_classes=K, y_train=ytr,
    default_train=dict(epochs=30, lr=5e-3, batch_size=64, weight_decay=0.0, patience=5, num_workers=0),
    collate_fn=pad_collate)
ft_summary.round(4)

In [ ]:
show_log("TF-IDF LR")[["exp_id", "description", "val_log_loss", "val_accuracy", "val_macro_f1"]]

## Final evaluation on the held-out test split (once) + Zindi submissions

In [ ]:
best_id = min(results_lr, key=lambda k: results_lr[k][2]["log_loss"])
e, Cval, _ = results_lr[best_id]
Xtr, Xte, Xz = tfidf_features(e["cfg"], tr.content, te.content, zindi.content)
clf = LogisticRegression(C=Cval, max_iter=3000, class_weight="balanced" if e["cfg"].get("balanced") else None).fit(Xtr, ytr)
save_final_results("baseline_tfidf_lr", "TF-IDF + LR", yte, clf.predict_proba(Xte), C.CLASSES, te.id.values,
                   extra={"selected_experiment": best_id, "C": Cval})
make_submission(zindi, clf.predict_proba(Xz), C.CLASSES, "baseline_tfidf_lr")

In [ ]:
_ = finalize(best_ft, build_model, TokenDataset(seq["te"], yte), yte, te.id.values, C.CLASSES,
         key="baseline_fasttext", display_name="fastText-style bag", collate_fn=pad_collate,
         zindi_ds=TokenDataset(seq["z"]), zindi_df=zindi, save_weights=False)

> **Write-up notes:** compare BL-01→BL-02 (morphology), BL-02→BL-03 (imbalance: check minority-class
> recall in `results/baseline_tfidf_lr/per_class.csv`, not just the averages), BL-04 vs BL-05 (where
> the signal is), BL-06 (stop-words). Fill the `observation` column in `experiments/log_tf_idf_lr.csv`.

In [ ]:
from src.utils import persist_outputs
persist_outputs()